# NIST StRD: nonlinear regression

Does Julia's nonlinear least squares reach NIST's certified estimates? The 27 nonlinear regression datasets of NIST's Statistical Reference Datasets, each fitted with LsqFit's `curve_fit` from NIST's two starting values, the first far from the solution and the second near it. NIST prints these certified values to 11 significant digits, so 10 or 11 is the most a fit can show.

The Statistical Reference Datasets (StRD) of the US National Institute of Standards and Technology (NIST) were built to test the numerical accuracy of statistical software. Each comes with values certified to 15 significant digits, computed in multiple precision. Some are real data; others are constructed to be hard, with many leading digits that do not change from one observation to the next, so that software which loses precision gives wrong digits. NIST grades each dataset's difficulty as lower, average or higher.

The data files sit beside this notebook as NIST publishes them (https://www.itl.nist.gov/div898/strd/), and every certified value is read from its file. The number of significant digits a value shares with NIST is the log relative error of McCullough (1998, *The American Statistician* 52(4), 358-366), the usual measure for these datasets: -log10(|value - certified| / |certified|), or -log10|value| when the certified value is 0, rounded down and at most 15.

In [ ]:
Pollis.packages("LsqFit")

## Set-up

`read_strd` reads a NIST file, `certified` reads a certified value from its header, `digits_same` counts the significant digits a value shares with it, `report` prints a dataset's values below NIST's and `summary` the fewest digits in each dataset.

In [ ]:
using LsqFit

using Printf

"""
	read_strd(name)

Read the NIST StRD file `name.dat` in this folder.

Gives: `data`, a matrix with one column per variable, the response first, and `header`, the text
above the data, which holds the certified values.
"""
function read_strd(name)
	lines = readlines(name * ".dat")
	range = match(r"Data\s*:?\s*\(?lines\s+(?<first>\d+)\s+to\s+(?<last>\d+)", join(lines[1:12], "\n"))
	first, last = parse(Int, range[:first]), parse(Int, range[:last])
	data = permutedims(reduce(hcat, [parse.(Float64, split(line)) for line in lines[first:last]]))
	return (data = data, header = join(lines[1:first - 1], "\n"))
end

"""
	certified(header, pattern)

The number captured as `v` by `pattern` in a file's header.
"""
certified(header, pattern) = parse(Float64, match(pattern, header)[:v])

"""
	digits_same(value, certified)

The number of significant digits `value` shares with `certified`: McCullough's log relative
error, or the log absolute error when `certified` is 0, rounded down and at most 15.
"""
function digits_same(value, certified)
	isfinite(value) || return 0
	error = certified == 0 ? abs(value) : abs(value - certified) / abs(certified)
	return error == 0 ? 15 : clamp(floor(Int, -log10(error)), 0, 15)
end

"""
	report(title, columns, rows)

Print each certified value with the values of `columns` below it, and the significant digits
each shares with NIST. Each row is `(label, certified, values...)`, one value for each column.

Gives: the fewest digits shared in each column.
"""
function report(title, columns, rows)
	println("\n", title)
	fewest = fill(15, length(columns))
	for (label, c, values...) in rows
		@printf("%-16s%-12s%25.15g\n", label, "certified", c)
		for (k, (column, v)) in enumerate(zip(columns, values))
			d = digits_same(v, c)
			fewest[k] = min(fewest[k], d)
			@printf("%-16s%-12s%25.15g  %s\n", "", column, v, d == 0 ? "DIFFERENT" : "same to $d significant digits")
		end
	end
	return fewest
end

"""
	summary(columns, results)

Print the fewest significant digits shared with NIST in each dataset, one column for each of
`columns`. Each result is `(name, level, fewest)`.
"""
function summary(columns, results)
	@printf("%-12s%-12s", "dataset", "difficulty")
	foreach(column -> @printf("%14s", column), columns)
	println()
	for (name, level, fewest) in results
		@printf("%-12s%-12s", name, level)
		foreach(d -> @printf("%14s", d == 0 ? "DIFFERENT" : string(d)), fewest)
		println()
	end
end

## 1. The datasets and their models

Each dataset with NIST's difficulty and its model, as written in its file, with `b` the parameters. Nelson's model is for the logarithm of its response and has two predictors, the columns of `x`.

In [ ]:
datasets = [
	"Misra1a" => ("lower", (x, b) -> b[1] .* (1 .- exp.(-b[2] .* x))),
	"Chwirut2" => ("lower", (x, b) -> exp.(-b[1] .* x) ./ (b[2] .+ b[3] .* x)),
	"Chwirut1" => ("lower", (x, b) -> exp.(-b[1] .* x) ./ (b[2] .+ b[3] .* x)),
	"Lanczos3" => ("lower", (x, b) -> b[1] .* exp.(-b[2] .* x) .+ b[3] .* exp.(-b[4] .* x) .+ b[5] .* exp.(-b[6] .* x)),
	"Gauss1" => ("lower", (x, b) -> b[1] .* exp.(-b[2] .* x) .+ b[3] .* exp.(-(x .- b[4]) .^ 2 ./ b[5]^2) .+ b[6] .* exp.(-(x .- b[7]) .^ 2 ./ b[8]^2)),
	"Gauss2" => ("lower", (x, b) -> b[1] .* exp.(-b[2] .* x) .+ b[3] .* exp.(-(x .- b[4]) .^ 2 ./ b[5]^2) .+ b[6] .* exp.(-(x .- b[7]) .^ 2 ./ b[8]^2)),
	"DanWood" => ("lower", (x, b) -> b[1] .* x .^ b[2]),
	"Misra1b" => ("lower", (x, b) -> b[1] .* (1 .- (1 .+ b[2] .* x ./ 2) .^ -2)),
	"Kirby2" => ("average", (x, b) -> (b[1] .+ b[2] .* x .+ b[3] .* x .^ 2) ./ (1 .+ b[4] .* x .+ b[5] .* x .^ 2)),
	"Hahn1" => ("average", (x, b) -> (b[1] .+ b[2] .* x .+ b[3] .* x .^ 2 .+ b[4] .* x .^ 3) ./ (1 .+ b[5] .* x .+ b[6] .* x .^ 2 .+ b[7] .* x .^ 3)),
	"Nelson" => ("average", (x, b) -> b[1] .- b[2] .* x[:, 1] .* exp.(-b[3] .* x[:, 2])),
	"MGH17" => ("average", (x, b) -> b[1] .+ b[2] .* exp.(-x .* b[4]) .+ b[3] .* exp.(-x .* b[5])),
	"Lanczos1" => ("average", (x, b) -> b[1] .* exp.(-b[2] .* x) .+ b[3] .* exp.(-b[4] .* x) .+ b[5] .* exp.(-b[6] .* x)),
	"Lanczos2" => ("average", (x, b) -> b[1] .* exp.(-b[2] .* x) .+ b[3] .* exp.(-b[4] .* x) .+ b[5] .* exp.(-b[6] .* x)),
	"Gauss3" => ("average", (x, b) -> b[1] .* exp.(-b[2] .* x) .+ b[3] .* exp.(-(x .- b[4]) .^ 2 ./ b[5]^2) .+ b[6] .* exp.(-(x .- b[7]) .^ 2 ./ b[8]^2)),
	"Misra1c" => ("average", (x, b) -> b[1] .* (1 .- (1 .+ 2 .* b[2] .* x) .^ -0.5)),
	"Misra1d" => ("average", (x, b) -> b[1] .* b[2] .* x ./ (1 .+ b[2] .* x)),
	"Roszman1" => ("average", (x, b) -> b[1] .- b[2] .* x .- atan.(b[3] ./ (x .- b[4])) ./ pi),
	"ENSO" => ("average", (x, b) -> b[1] .+ b[2] .* cos.(2pi .* x ./ 12) .+ b[3] .* sin.(2pi .* x ./ 12) .+
		b[5] .* cos.(2pi .* x ./ b[4]) .+ b[6] .* sin.(2pi .* x ./ b[4]) .+ b[8] .* cos.(2pi .* x ./ b[7]) .+ b[9] .* sin.(2pi .* x ./ b[7])),
	"MGH09" => ("higher", (x, b) -> b[1] .* (x .^ 2 .+ x .* b[2]) ./ (x .^ 2 .+ x .* b[3] .+ b[4])),
	"Thurber" => ("higher", (x, b) -> (b[1] .+ b[2] .* x .+ b[3] .* x .^ 2 .+ b[4] .* x .^ 3) ./ (1 .+ b[5] .* x .+ b[6] .* x .^ 2 .+ b[7] .* x .^ 3)),
	"BoxBOD" => ("higher", (x, b) -> b[1] .* (1 .- exp.(-b[2] .* x))),
	"Rat42" => ("higher", (x, b) -> b[1] ./ (1 .+ exp.(b[2] .- b[3] .* x))),
	"MGH10" => ("higher", (x, b) -> b[1] .* exp.(b[2] ./ (x .+ b[3]))),
	"Eckerle4" => ("higher", (x, b) -> (b[1] / b[2]) .* exp.(-0.5 .* ((x .- b[3]) ./ b[2]) .^ 2)),
	"Rat43" => ("higher", (x, b) -> b[1] ./ (1 .+ exp.(b[2] .- b[3] .* x)) .^ (1 / b[4])),
	"Bennett5" => ("higher", (x, b) -> b[1] .* (b[2] .+ x) .^ (-1 / b[3])),
]

## 2. Fit from both starting values

`curve_fit` minimises the sum of squared residuals by Levenberg-Marquardt, with derivatives by automatic differentiation (ForwardDiff). It stops when a step changes the parameters by less than `x_tol = 1e-8`, the gradient is below `g_tol = 1e-12`, or after `maxIter = 1000` iterations. Each dataset is fitted from both starting values twice: with these defaults, and with tolerances so tight (`tight`) that it stops only where Float64 can go no further. A fit that fails, or stops where the standard errors cannot be computed, gives `NaN`, shown as DIFFERENT. Below each dataset, `converged` is `curve_fit`'s own verdict on each fit.

In [ ]:
"""
	nls_fit(model, x, y, start; options...)

Fit `model` to `y` from `start` with `curve_fit`, passing it `options`.

Gives: the estimates, standard errors, residual sum of squares and residual standard deviation,
`NaN` where they cannot be computed, and whether `curve_fit` reports convergence.
"""
function nls_fit(model, x, y, start; options...)
	failed = (coef = fill(NaN, length(start)), se = fill(NaN, length(start)), rss = NaN, sd = NaN, converged = false)
	fit = try
		curve_fit(model, x, y, start; options...)
	catch
		return failed
	end
	rss = sum(abs2, fit.resid)
	se = try
		stderror(fit)
	catch
		fill(NaN, length(start))
	end
	return (coef = coef(fit), se = se, rss = rss, sd = sqrt(rss / (length(y) - length(start))), converged = fit.converged)
end

tight = (x_tol = 1e-15, g_tol = 1e-15, maxIter = 100_000)  # stop only at the limit of Float64
columns = ["start 1", "start 2", "1, tight", "2, tight"]
results = []
for (name, (level, model)) in datasets
	(; data, header) = read_strd(name)
	y = name == "Nelson" ? log.(data[:, 1]) : data[:, 1]
	x = size(data, 2) == 2 ? data[:, 2] : data[:, 2:end]
	table = [parse.(Float64, m.captures) for m in eachmatch(r"\n *b\d+ *= *(?<start1>\S+) +(?<start2>\S+) +(?<estimate>\S+) +(?<se>\S+)", header)]
	starts = [[t[s] for t in table] for s in 1:2]
	fits = [[nls_fit(model, x, y, start) for start in starts]; [nls_fit(model, x, y, start; tight...) for start in starts]]
	rows = []
	for (i, (_, _, estimate, se)) in enumerate(table)
		push!(rows, ("b$i", estimate, (f.coef[i] for f in fits)...))
		push!(rows, ("  SE", se, (f.se[i] for f in fits)...))
	end
	push!(rows, ("RSS", certified(header, r"Residual Sum of Squares: +(?<v>\S+)"), (f.rss for f in fits)...))
	push!(rows, ("residual SD", certified(header, r"Residual Standard Deviation: +(?<v>\S+)"), (f.sd for f in fits)...))
	fewest = report("$name ($level difficulty, $(length(y)) observations)", columns, rows)
	println("converged       ", join(("$column: $(f.converged)" for (column, f) in zip(columns, fits)), ", "))
	push!(results, (name, level, fewest))
end

## 3. Summary

The fewest significant digits shared with NIST in each dataset, from each starting value, with the default and the tight tolerances.

In [ ]:
summary(columns, results)

## What the summary shows

From the second starting value, near the solution, `curve_fit` with its defaults gives 6 to 10 digits on every dataset but two. Bennett5 gives 1: its fit creeps so slowly that it stops at the 1,000-iteration limit, not converged; with the tight tolerances it converges and gives 8, and most other datasets gain a digit or two. Lanczos1 gives 2 or 3 digits from every start, and no method could do better in Float64: its data follow the model to about 13 digits, so its residuals are near 1e-13 on values up to 2.5, and Float64, with about 16 significant digits, keeps only 3 of theirs. Its residual sum of squares and standard deviation, and the standard errors computed from them, carry only those 3 digits.

From the first starting value, far from the solution, MGH17, BoxBOD and MGH10 fail with either setting. MGH17 and BoxBOD stop at a point that is not the solution, with a residual sum of squares of 1.106 against NIST's 0.0000546 and 9771.5 against 1168.0, yet `curve_fit` reports them converged. MGH10 does not converge, and with tight tolerances its parameters overflow. Tighter tolerances cannot rescue a fit that heads for the wrong place: compare the residual sum of squares from several starting values, and never trust a converged flag alone.